# Donor Lapse/Churn Intelligence — (3 of 3) Serving, Observability & Agent

**Notebook 3 of 3.** Scores the donor base through the **Feature Store** (fresh, point-in-time features), sets up **ML Observability**, wraps the model in SQL **tool functions**, and registers a **Cortex Agent** that retrieves donors via Cortex Analyst and scores them with the deployed model.

> Run **`donor-churn-01-features.ipynb`** and **`donor-churn-02-model.ipynb`** first.

---
## Rehydrate session & objects

Reconnect and get handles to the Feature Store views and the DEFAULT registered model.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.registry import Registry

session = get_active_session()
for stmt in ["USE ROLE ACCOUNTADMIN", "USE DATABASE DONOR_CHURN_ML_DEMO",
             "USE SCHEMA MODELS", "USE WAREHOUSE DONOR_CHURN_ML_WH"]:
    session.sql(stmt).collect()

fs = FeatureStore(session=session, database="DONOR_CHURN_ML_DEMO", name="FEATURES",
                  default_warehouse="DONOR_CHURN_ML_WH", creation_mode=CreationMode.FAIL_IF_NOT_EXIST)
donor_rfm_fv = fs.get_feature_view("DONOR_RFM_FV", "v1")
donor_eng_fv = fs.get_feature_view("DONOR_ENGAGEMENT_FV", "v1")
donor_static_fv = fs.get_feature_view("DONOR_STATIC_FV", "v1")

reg = Registry(session=session, database_name="DONOR_CHURN_ML_DEMO", schema_name="MODELS")
mv = reg.get_model("DONOR_LAPSE_MODEL").default
print("Using model version:", mv.version_name)

FEATURES = [
    "WEALTH_SIGNAL_SCORE", "TENURE_DAYS", "FREQUENCY_LIFETIME", "FREQUENCY_LAST_12M",
    "MONETARY_TOTAL", "AVG_GIFT_AMOUNT", "RECENCY_DAYS", "ENGAGEMENT_COUNT", "ENGAGEMENT_LAST_6M",
]

def positive_proba_col(scored_df):
    proba_cols = [c for c in scored_df.columns if "PROBA" in c.upper()]
    if not proba_cols:
        raise ValueError("No predict_proba output columns found: " + str(scored_df.columns))
    for c in proba_cols:
        if c.upper().rstrip('"').endswith("1"):
            return c
    return proba_cols[-1]

---
## Section 9: Model Serving

Score the whole donor base in **batch**, and a **single donor** in real time — both retrieve features **through the Feature Store** at `CURRENT_TIMESTAMP()`, so serving uses the *latest* snapshot of the same definitions used in training (no train/serve skew). We store the **churn probability** (`predict_proba`), not a hard class.

In [ ]:
# Batch scoring — fresh features via point-in-time Feature Store retrieval.
spine_df = session.sql("""
    SELECT donor_id, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS as_of_ts
    FROM DONOR_CHURN_ML_DEMO.RAW.DONORS
""")

all_features = fs.retrieve_feature_values(
    spine_df=spine_df,
    features=[donor_rfm_fv, donor_eng_fv, donor_static_fv],
    spine_timestamp_col="AS_OF_TS",
)

scored = mv.run(all_features.select("DONOR_ID", *FEATURES), function_name="predict_proba")
pcol = positive_proba_col(scored)
scored.select("DONOR_ID", F.col(pcol).alias("PRED_LAPSE")) \
      .write.mode("overwrite").save_as_table("DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES")

session.sql("""
    SELECT donor_id, ROUND(PRED_LAPSE, 4) AS churn_probability
    FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES
    ORDER BY PRED_LAPSE DESC LIMIT 10
""").show()

In [ ]:
# Single-donor real-time scoring — dynamically pick the current highest-risk donor.
top_id = session.sql("""
    SELECT donor_id FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES
    ORDER BY PRED_LAPSE DESC LIMIT 1
""").collect()[0]["DONOR_ID"]
print("Scoring donor_id:", top_id)

one_spine = session.sql(f"""
    SELECT donor_id, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS as_of_ts
    FROM DONOR_CHURN_ML_DEMO.RAW.DONORS WHERE donor_id = {top_id}
""")
one_features = fs.retrieve_feature_values(spine_df=one_spine,
    features=[donor_rfm_fv, donor_eng_fv, donor_static_fv], spine_timestamp_col="AS_OF_TS")
one_scored = mv.run(one_features.select("DONOR_ID", *FEATURES), function_name="predict_proba")
one_scored.select("DONOR_ID", F.col(positive_proba_col(one_scored)).alias("CHURN_PROBABILITY")).show()

---
## Section 10: ML Observability

Create a **model monitor** tracking prediction drift, data drift, and accuracy over time — segmented by region. Constraints: `TIMESTAMP_NTZ` timestamp, `NUMBER` prediction/actual columns, monitor in the same schema as the model, no NULLs. The prediction column holds the **churn probability**.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG AS
    SELECT b.donor_id, b.region,
           CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS scored_at,
           b.wealth_signal_score, b.tenure_days, b.frequency_lifetime,
           b.frequency_last_12m, b.monetary_total, b.avg_gift_amount,
           b.recency_days, b.engagement_count, b.engagement_last_6m,
           s.PRED_LAPSE::NUMBER(10,6) AS pred_lapse,
           b.is_lapsed::NUMBER        AS actual_lapse
    FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b
    JOIN DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s USING (donor_id)
""").collect()
session.sql("""
    CREATE OR REPLACE TABLE DONOR_CHURN_ML_DEMO.MODELS.LAPSE_BASELINE_SNAPSHOT AS
    SELECT * FROM DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG
""").collect()
print("Scoring log + baseline snapshot ready.")

In [ ]:
session.sql("""
    CREATE OR REPLACE MODEL MONITOR DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_MONITOR WITH
        MODEL = DONOR_LAPSE_MODEL
        VERSION = V1
        FUNCTION = PREDICT_PROBA
        SOURCE = DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG
        BASELINE = DONOR_CHURN_ML_DEMO.MODELS.LAPSE_BASELINE_SNAPSHOT
        WAREHOUSE = DONOR_CHURN_ML_WH
        REFRESH_INTERVAL = '1 day'
        AGGREGATION_WINDOW = '1 day'
        TIMESTAMP_COLUMN = scored_at
        ID_COLUMNS = ('DONOR_ID')
        PREDICTION_SCORE_COLUMNS = ('PRED_LAPSE')
        ACTUAL_CLASS_COLUMNS = ('ACTUAL_LAPSE')
        SEGMENT_COLUMNS = ('REGION')
""").collect()
print("Model monitor created. Query drift with MODEL_MONITOR_DRIFT_METRIC:")
session.sql("""
    SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        'DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_MONITOR',
        'DIFFERENCE_OF_MEANS', 'PRED_LAPSE', '1 DAY',
        DATEADD('day', -30, CURRENT_DATE()), CURRENT_DATE()))
""").show()

In [ ]:
# Reference alert (needs an email notification integration). Adjust names before enabling.
print("""
CREATE OR REPLACE ALERT DONOR_CHURN_ML_DEMO.MODELS.LAPSE_MODEL_ACCURACY_ALERT
  WAREHOUSE = DONOR_CHURN_ML_WH
  SCHEDULE = '1440 MINUTE'
  IF (EXISTS (
      SELECT 1 FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
          'DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_MONITOR',
          'F1', '1 DAY', DATEADD('day', -1, CURRENT_DATE()), CURRENT_DATE()))
      WHERE METRIC_VALUE < 0.80))
  THEN CALL SYSTEM$SEND_EMAIL('my_email_int','team@example.org',
       'Donor lapse model F1 dropped below 0.80', 'Investigate drift in the monitor dashboard.');
""")

---
## Section 11: Tool Wrappers (model as a callable tool)

Wrap the deployed model in SQL so an agent can call it: `TOP_CHURN_RISK(segment, n)` ranks the highest-risk donors in a segment; `PREDICT_DONOR_CHURN(donor_id)` scores one donor. Each returns churn probability, top risk drivers, and (for the batch variant) a recommended action via `AI_COMPLETE`.

In [ ]:
# Drop any pre-existing procedure of the same name (object-type conflict with the function)
session.sql("DROP PROCEDURE IF EXISTS DONOR_CHURN_ML_DEMO.MODELS.TOP_CHURN_RISK(STRING, NUMBER)").collect()
# NOTE: a SQL table UDF body cannot use a function argument in LIMIT, so top-N is expressed
# with QUALIFY ROW_NUMBER() <= N. AI_COMPLETE returns VARIANT, cast to STRING to match the
# declared RETURNS TABLE column type.
session.sql("""
    CREATE OR REPLACE FUNCTION DONOR_CHURN_ML_DEMO.MODELS.TOP_CHURN_RISK(SEGMENT STRING, N NUMBER)
    RETURNS TABLE (donor_id NUMBER, region STRING, churn_probability FLOAT,
                   risk_drivers STRING, recommended_action STRING)
    AS
    $$
        SELECT s.donor_id, b.region,
               ROUND(s.PRED_LAPSE, 4) AS churn_probability,
               'recency ' || b.recency_days || 'd; gifts_12m ' || b.frequency_last_12m
                 || '; engagement_6m ' || b.engagement_last_6m AS risk_drivers,
               SNOWFLAKE.CORTEX.AI_COMPLETE('llama3.1-8b',
                 'A ' || b.donor_segment || ' donor in ' || b.region
                 || ' has churn probability ' || ROUND(s.PRED_LAPSE,2)
                 || '. Last gift ' || b.recency_days || ' days ago, ' || b.frequency_last_12m
                 || ' gifts in the last year, ' || b.engagement_last_6m
                 || ' engagements in 6 months. Recommend one concise next-best retention action.'
               )::STRING AS recommended_action
        FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s
        JOIN DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b USING (donor_id)
        WHERE b.donor_segment = SEGMENT
        QUALIFY ROW_NUMBER() OVER (ORDER BY s.PRED_LAPSE DESC) <= N
    $$;
""").collect()
print("Created TOP_CHURN_RISK. Example (top 5 Major donors):")
session.sql("SELECT * FROM TABLE(DONOR_CHURN_ML_DEMO.MODELS.TOP_CHURN_RISK('Major', 5))").show()

In [ ]:
# The parameter is named P_DONOR_ID to avoid colliding with the donor_id column and
# to reference it directly (a UDF argument cannot be qualified by the function name).
session.sql("""
    CREATE OR REPLACE FUNCTION DONOR_CHURN_ML_DEMO.MODELS.PREDICT_DONOR_CHURN(P_DONOR_ID NUMBER)
    RETURNS TABLE (donor_id NUMBER, churn_probability FLOAT, risk_drivers STRING)
    AS
    $$
        SELECT s.donor_id, ROUND(s.PRED_LAPSE, 4) AS churn_probability,
               'recency ' || b.recency_days || 'd; gifts_12m ' || b.frequency_last_12m
                 || '; engagement_6m ' || b.engagement_last_6m AS risk_drivers
        FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s
        JOIN DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b USING (donor_id)
        WHERE s.donor_id = P_DONOR_ID
    $$;
""").collect()
top_id = session.sql("SELECT donor_id FROM DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES ORDER BY PRED_LAPSE DESC LIMIT 1").collect()[0]["DONOR_ID"]
session.sql(f"SELECT * FROM TABLE(DONOR_CHURN_ML_DEMO.MODELS.PREDICT_DONOR_CHURN({top_id}))").show()

---
## Section 12: Cortex Agent (model as a tool)

Register an agent with **two tools**: Cortex Analyst (retrieval over the semantic view) and a **custom tool** backed by `TOP_CHURN_RISK` (model scoring + explanation). The agent plans: retrieve a segment via Analyst → score via the model tool → return an explained, ranked action list.

> The `Score_Donor_Churn` custom tool (`type: generic`) is bound **directly to the `TOP_CHURN_RISK` function** via `tool_resources` (`identifier` + `type: function` + `execution_environment`), and its `input_schema` declares the `SEGMENT` and `N` arguments. The agent can therefore call it natively — no Snowsight UI binding step is required. If the role that runs the agent is not the function's owner, grant `USAGE` on `TOP_CHURN_RISK` to that role.

In [ ]:
session.sql("""
    CREATE OR REPLACE AGENT DONOR_CHURN_ML_DEMO.MODELS.DONOR_RETENTION_AGENT
      COMMENT = 'Donor retention agent: Analyst retrieval + deployed lapse model scoring'
      PROFILE = '{"display_name": "Donor Retention Intelligence", "color": "blue"}'
      FROM SPECIFICATION
      $$
models:
  orchestration: auto
instructions:
  orchestration: |
    ## Role
    You are the Donor Retention Intelligence agent for a nonprofit fundraising team
    (development / advancement officers and their leadership). You help them find which
    donors are at risk of lapsing, understand why, and decide the next-best retention action.

    ## Domain context
    - "Lapse" / "churn" = a donor stops giving (no gift within the forward label window).
    - Donor segments are exactly: "Major" (major-gift donors), "Mid", "Grassroots".
    - Regions are exactly: West, Northeast, Southeast, Midwest, Southwest.
    - Two DIFFERENT risk concepts exist; do not conflate them:
      * lapse_rate  = a HISTORICAL aggregate (share of donors that already lapsed) from Donor_Metrics.
      * churn_probability = the deployed MODEL's forward-looking predicted probability for an
        individual donor (0-1), from Score_Donor_Churn.
    - Model risk drivers are RFM + engagement signals: recency (days since last gift),
      gifts in last 12 months, engagements in last 6 months, tenure, wealth signal.

    ## Tool selection
    - Use Donor_Metrics (Cortex Analyst) for COUNTS, SUMS, AVERAGES, RATES, and BREAKDOWNS by
      dimension: donor counts, total/average giving, gift counts, engagement counts, and the
      historical lapse_rate — sliced by region, segment, channel, appeal, event type, or month.
    - Use Score_Donor_Churn to RANK and EXPLAIN individual at-risk donors with the deployed
      model: it returns the top-N highest-risk donors in one segment with churn probability,
      risk drivers, and a recommended action.
    - Do NOT use Donor_Metrics to predict a single donor's risk or to get recommended actions.
    - Do NOT use Score_Donor_Churn to compute aggregates or rates.

    ## Important tool constraint (Score_Donor_Churn)
    Score_Donor_Churn filters by SEGMENT ONLY and takes a count N. It does NOT accept a region
    or any other filter. To answer a region- or otherwise-narrowed request:
      1. Call Score_Donor_Churn with the requested segment and a LARGER N (e.g., 25-50).
      2. Filter the returned rows on the "region" column (or other returned fields) yourself.
      3. Rank by churn_probability and present the requested number.

    ## Flagship workflow — "who is most at risk and what should we do"
    For questions like "which <segment> donors in <region> are most at risk and what should we do":
      1. Score_Donor_Churn(segment=<segment>, n=~30) to get scored, explained candidates.
      2. Keep only rows matching the requested region.
      3. Rank by churn_probability (desc), present the top few with drivers + recommended action.
    If a request needs a segment- or region-level RATE alongside individuals, also call
    Donor_Metrics for the lapse_rate to add context.

    ## Boundaries
    - This is synthetic demo data for a generic nonprofit CRM; there are no real donor identities.
    - Only these segments/regions exist (listed above). If asked about others, say so and offer the valid values.
    - You analyze and recommend; you do not send outreach or modify records.
  response: |
    - Be concise and lead with the direct answer.
    - Express churn_probability and lapse_rate as percentages (e.g., 0.82 -> "82%").
    - Show monetary values with a currency symbol.
    - When ranking donors, use a table with: donor_id, region, churn probability (%), top risk
      drivers, and the recommended retention action.
    - For a single metric or rate, state it in one sentence (with the dimension/segment/region).
    - Make clear when a number is the model's predicted churn probability vs. the historical lapse rate.
  sample_questions:
    - question: "Which major-gift donors in the West region are most at risk of lapsing, why, and what should we do?"
    - question: "What is the lapse rate by region?"
    - question: "Score the top 10 Grassroots donors most likely to lapse and recommend an action for each."
    - question: "How do total giving and average gift compare across donor segments?"
tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "Donor_Metrics"
      description: |
        Text-to-SQL over the governed DONOR_CHURN_SV semantic view. Answers aggregate and
        breakdown questions about donors, donations, engagement, and HISTORICAL lapse behavior.

        Metrics: donor_count, total_giving (currency), avg_gift (currency), gift_count,
        engagement_count, lapsed_donors, lapse_rate (share of donors that lapsed, 0-1).
        Dimensions (valid values):
          - region: West, Northeast, Southeast, Midwest, Southwest
          - donor_segment: Major, Mid, Grassroots (Major = major-gift donors)
          - channel (acquisition channel), appeal_type, event_type
          - acquisition_month, gift_month (month grains)

        When to use:
          - Counts, sums, averages, rates, rankings, and breakdowns by any dimension above.
          - Historical lapse_rate by region, segment, etc.
          - Comparisons across segments/regions or over months.
        When NOT to use:
          - Do NOT use to predict an individual donor's churn risk, get model risk drivers, or a
            recommended action -> use Score_Donor_Churn.
          - lapse_rate here is a HISTORICAL aggregate, not the model's predicted probability.

        Query tips: name the metric and the dimension explicitly and use the exact segment/region
        values above. Examples: "lapse_rate by region"; "total_giving and avg_gift by donor_segment";
        "lapse_rate for Major donors in the West"; "engagement_count by event_type for the last 6 months".
  - tool_spec:
      type: "generic"
      name: "Score_Donor_Churn"
      description: |
        Runs the deployed lapse/churn model to return the N highest-risk donors within ONE donor
        segment, each with the model's forward-looking churn probability, top risk drivers, and an
        AI-generated next-best retention action. Backed by TOP_CHURN_RISK(segment, n).

        Parameters:
          - segment (string, required): exactly one of "Major", "Mid", "Grassroots" (case-sensitive).
            "Major" = major-gift donors.
          - n (integer, required): number of highest-risk donors to return, ranked by churn
            probability descending (e.g., 10). Use a larger n (25-50) when you must post-filter the
            results by region or another attribute.
        Returns: donor_id, region, churn_probability (0-1 model probability), risk_drivers
          (recency / gifts-12m / engagement-6m), recommended_action.

        When to use:
          - "Who is most likely to lapse", "top at-risk donors", "score these donors and tell me why /
            what to do" for a given segment.
        When NOT to use / constraints:
          - It filters by SEGMENT ONLY and cannot take a region or any other filter. For a region- or
            attribute-specific ask, request a larger n and filter the returned rows yourself.
          - Do NOT use for aggregates, counts, or rates -> use Donor_Metrics.
          - churn_probability is the model's prediction, distinct from the historical lapse_rate.
      input_schema:
        type: object
        properties:
          SEGMENT:
            type: string
            description: "Donor segment to score. Exactly one of Major, Mid, or Grassroots (case-sensitive). Major = major-gift donors."
          N:
            type: integer
            description: "Number of highest-risk donors to return, ranked by churn probability descending. Use a larger value (25-50) when you must post-filter results by region or another attribute."
        required:
          - SEGMENT
          - N
tool_resources:
  Donor_Metrics:
    semantic_view: "DONOR_CHURN_ML_DEMO.ANALYTICS.DONOR_CHURN_SV"
    execution_environment:
      type: "warehouse"
      warehouse: "DONOR_CHURN_ML_WH"
  Score_Donor_Churn:
    identifier: "DONOR_CHURN_ML_DEMO.MODELS.TOP_CHURN_RISK"
    type: "function"
    execution_environment:
      type: "warehouse"
      warehouse: "DONOR_CHURN_ML_WH"
      $$
""").collect()
print("Agent DONOR_RETENTION_AGENT created with BOTH tools bound (Donor_Metrics + Score_Donor_Churn).")
print("Score_Donor_Churn is wired directly to TOP_CHURN_RISK(segment, n) via tool_resources -- no Snowsight step needed.")
print("If the agent's run role differs from the function owner, grant USAGE on TOP_CHURN_RISK to that role.")

---
## Section 13: The "Wow" Moment & Summary

Chat with the agent (Snowsight **AI & ML → Agents**, or `app/streamlit_app.py`):

1. *"Which of our major-gift donors in the West region are most at risk of lapsing this quarter, why, and what should we do?"*
2. *"Draft outreach for the top 3."*

### What you built (across all 3 notebooks)

| Capability | Object |
|---|---|
| Feature Store | `DONOR` entity + managed PIT views `DONOR_RFM_FV`, `DONOR_ENGAGEMENT_FV`, `DONOR_STATIC_FV` |
| Dataset | `LAPSE_TRAINING_SET/v1` |
| Cortex ML Functions | `DONATION_VOLUME_FORECAST`, `DONATION_VOLUME_ANOMALY`, `LAPSE_BASELINE_MODEL` |
| Experiment Tracking | experiment `DONOR_LAPSE` with per-config runs |
| Snowpark ML + HPO | `XGBClassifier` + `GridSearchCV` |
| ML Jobs | `@remote` training on `DONOR_CHURN_ML_POOL` |
| Registry | `DONOR_LAPSE_MODEL/v1` (DEFAULT, explainability on) |
| Explainability | Shapley via `mv.run(function_name='explain')` |
| Serving | `DONOR_LAPSE_SCORES` (batch, probability) + single-donor |
| Observability | `DONOR_LAPSE_MONITOR` + metric functions + alert |
| Tool wrappers | `TOP_CHURN_RISK`, `PREDICT_DONOR_CHURN` |
| Agent | `DONOR_RETENTION_AGENT` (Analyst + model tool) |
| Orchestration *(optional, below)* | `DONOR_LAPSE_TRAINING_DAG` |

Everything ran inside one governed Snowflake account — no data movement.

---
## Section 14 (Optional Extension): Orchestrate Training as a Task Graph (DAG)

To *productionize* the pipeline, wrap the steps in a **task graph (DAG)** so retraining runs on a schedule, every stage gets retries + run history, and you can watch it in **Snowsight → Monitoring → Task History / Graphs**.

```
PREP_TRAINING_DATA  →  TRAIN_AND_REGISTER  →  BATCH_SCORE  →  REFRESH_MONITOR
```

The training stage is a **Snowflake ML Job** on the compute pool that registers a fresh model version — orchestrated natively by the task. Built with `snowflake.core.task.dagv1`.

> **Requirements:** the `DONOR_CHURN_ML_POOL` compute pool, `snowflake-ml-python >= 1.26`, and a role that can `CREATE TASK` / `EXECUTE TASK`. Optional — skip if your role can't create compute pools or tasks.

In [ ]:
# 14a. Define the four pipeline stages.
from snowflake.snowpark import Session
from snowflake.ml.jobs import remote

session.sql("USE SCHEMA MODELS").collect()
session.sql("CREATE STAGE IF NOT EXISTS DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE").collect()
try:
    session.sql("""
        CREATE COMPUTE POOL IF NOT EXISTS DONOR_CHURN_ML_POOL
          MIN_NODES = 1 MAX_NODES = 2 INSTANCE_FAMILY = CPU_X64_M
    """).collect()
except Exception as e:
    print("Compute pool step skipped (needs CREATE COMPUTE POOL privilege):", e)

FEATS = ["WEALTH_SIGNAL_SCORE","TENURE_DAYS","FREQUENCY_LIFETIME","FREQUENCY_LAST_12M",
         "MONETARY_TOTAL","AVG_GIFT_AMOUNT","RECENCY_DAYS","ENGAGEMENT_COUNT","ENGAGEMENT_LAST_6M"]
TRAINING_TABLE = "DONOR_CHURN_ML_DEMO.FEATURES.LAPSE_TRAINING_CURRENT"

def prep_training_data(session: Session) -> str:
    session.sql(f"""
        CREATE OR REPLACE TABLE {TRAINING_TABLE} AS
        SELECT donor_id, wealth_signal_score, tenure_days, frequency_lifetime,
               frequency_last_12m, monetary_total, avg_gift_amount, recency_days,
               engagement_count, engagement_last_6m, is_lapsed
        FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE
    """).collect()
    return "prepared " + TRAINING_TABLE

@remote("DONOR_CHURN_ML_POOL", stage_name="DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE")
def train_and_register(training_table: str = TRAINING_TABLE):
    from snowflake.snowpark.context import get_active_session
    from snowflake.ml.modeling.xgboost import XGBClassifier
    from snowflake.ml.registry import Registry
    s = get_active_session()
    feats = ["WEALTH_SIGNAL_SCORE","TENURE_DAYS","FREQUENCY_LIFETIME","FREQUENCY_LAST_12M",
             "MONETARY_TOTAL","AVG_GIFT_AMOUNT","RECENCY_DAYS","ENGAGEMENT_COUNT","ENGAGEMENT_LAST_6M"]
    df = s.table(training_table)
    train_df, _ = df.random_split([0.8, 0.2], seed=42)
    model = XGBClassifier(input_cols=feats, label_cols=["IS_LAPSED"], output_cols=["PRED_LAPSE"],
                          n_estimators=400, max_depth=6, learning_rate=0.05)
    model.fit(train_df)
    reg = Registry(session=s, database_name="DONOR_CHURN_ML_DEMO", schema_name="MODELS")
    mv = reg.log_model(model=model, model_name="DONOR_LAPSE_MODEL",
                       sample_input_data=train_df.select(feats).limit(100),
                       comment="Retrained by DONOR_LAPSE_TRAINING_DAG",
                       options={"enable_explainability": True},
                       target_platforms=["WAREHOUSE"])
    reg.get_model("DONOR_LAPSE_MODEL").default = mv.version_name
    return mv.version_name

def batch_score(session: Session) -> str:
    from snowflake.ml.registry import Registry
    from snowflake.snowpark import functions as F
    feats = ["WEALTH_SIGNAL_SCORE","TENURE_DAYS","FREQUENCY_LIFETIME","FREQUENCY_LAST_12M",
             "MONETARY_TOTAL","AVG_GIFT_AMOUNT","RECENCY_DAYS","ENGAGEMENT_COUNT","ENGAGEMENT_LAST_6M"]
    reg = Registry(session=session, database_name="DONOR_CHURN_ML_DEMO", schema_name="MODELS")
    mv = reg.get_model("DONOR_LAPSE_MODEL").default
    src = session.sql("SELECT donor_id, " + ", ".join(feats) + " FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE")
    scored = mv.run(src, function_name="predict_proba")
    pcol = [c for c in scored.columns if "PROBA" in c.upper() and c.upper().rstrip('"').endswith("1")][0]
    scored.select("DONOR_ID", F.col(pcol).alias("PRED_LAPSE")).write.mode("overwrite").save_as_table(
        "DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES")
    return "scored to DONOR_LAPSE_SCORES (version " + str(mv.version_name) + ")"

def refresh_monitor(session: Session) -> str:
    session.sql("""
        CREATE OR REPLACE TABLE DONOR_CHURN_ML_DEMO.MODELS.LAPSE_SCORING_LOG AS
        SELECT b.donor_id, b.region,
               CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS scored_at,
               b.wealth_signal_score, b.tenure_days, b.frequency_lifetime,
               b.frequency_last_12m, b.monetary_total, b.avg_gift_amount,
               b.recency_days, b.engagement_count, b.engagement_last_6m,
               s.PRED_LAPSE::NUMBER(10,6) AS pred_lapse,
               b.is_lapsed::NUMBER        AS actual_lapse
        FROM DONOR_CHURN_ML_DEMO.RAW.DONOR_TRAINING_BASE b
        JOIN DONOR_CHURN_ML_DEMO.MODELS.DONOR_LAPSE_SCORES s USING (donor_id)
    """).collect()
    return "monitor source refreshed"

print("Defined pipeline stages: prep -> train_and_register (ML Job) -> batch_score -> refresh_monitor")

In [ ]:
# 14b. Assemble the task graph with the Snowflake Python API and deploy it.
from datetime import timedelta
from snowflake.core import Root
from snowflake.core._common import CreateMode
from snowflake.core.task import StoredProcedureCall
from snowflake.core.task.dagv1 import DAG, DAGTask, DAGOperation

root = Root(session)
STAGE = "@DONOR_CHURN_ML_DEMO.MODELS.PAYLOAD_STAGE"
WH = "DONOR_CHURN_ML_WH"
PKGS = ["snowflake-snowpark-python", "snowflake-ml-python"]

with DAG("DONOR_LAPSE_TRAINING_DAG", schedule=timedelta(days=7), warehouse=WH,
         stage_location=STAGE, packages=PKGS) as dag:
    t_prep = DAGTask("PREP_TRAINING_DATA",
        definition=StoredProcedureCall(prep_training_data, stage_location=STAGE, packages=PKGS), warehouse=WH)
    t_train = DAGTask("TRAIN_AND_REGISTER", definition=train_and_register)
    t_score = DAGTask("BATCH_SCORE",
        definition=StoredProcedureCall(batch_score, stage_location=STAGE, packages=PKGS), warehouse=WH)
    t_monitor = DAGTask("REFRESH_MONITOR",
        definition=StoredProcedureCall(refresh_monitor, stage_location=STAGE, packages=PKGS), warehouse=WH)
    t_prep >> t_train >> t_score >> t_monitor

schema = root.databases["DONOR_CHURN_ML_DEMO"].schemas["MODELS"]
dag_op = DAGOperation(schema)
dag_op.deploy(dag, mode=CreateMode.or_replace)
print("Deployed task graph DONOR_LAPSE_TRAINING_DAG (tasks start suspended).")

In [ ]:
# 14c. Resume every task in the graph, then trigger one manual run.
graph_tasks = root.databases["DONOR_CHURN_ML_DEMO"].schemas["MODELS"].tasks
for t in graph_tasks.iter(like="DONOR_LAPSE_TRAINING_DAG%"):
    try:
        graph_tasks[t.name].resume()
    except Exception as e:
        print("resume skipped for", t.name, "->", e)

dag_op.run(dag)
print("Triggered DONOR_LAPSE_TRAINING_DAG. Watch it live: Snowsight -> Monitoring -> Task History (Graph view).")

In [ ]:
# 14d. Inspect the run programmatically.
session.sql("""
    SELECT name, state, scheduled_time, completed_time, error_message
    FROM TABLE(DONOR_CHURN_ML_DEMO.INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP())))
    WHERE name LIKE 'DONOR_LAPSE_TRAINING_DAG%'
    ORDER BY scheduled_time DESC, name
""").show()

### Operate & clean up

```python
# Suspend the schedule (keeps the graph defined; stops future runs):
root.databases["DONOR_CHURN_ML_DEMO"].schemas["MODELS"] \
    .tasks["DONOR_LAPSE_TRAINING_DAG$PREP_TRAINING_DATA"].suspend()

# ...or drop the whole task graph:
dag_op.drop(dag)

# The compute pool keeps billing while running — suspend or drop it if unused:
session.sql("ALTER COMPUTE POOL DONOR_CHURN_ML_POOL SUSPEND").collect()
```